# 📗 부록 01: pytest로 테스트 작성하고 실행하기

교안 02에서는 평가 결과 파일을 pytest 테스트로 판정해 회귀 게이트를 만들었습니다. 이 부록은 그 바탕인 단위 테스트와 pytest를 짧게 익힙니다.

**단위 테스트**(unit test)는 함수 하나처럼 작은 코드 단위에 정해 둔 입력을 넣고, 나와야 할 결과가 나오는지 코드로 확인하는 것입니다. 한 번 적어 두면 코드를 고칠 때마다 몇 초 만에 같은 기준으로 다시 확인할 수 있습니다.

| 실무에서 쓰는 이유 | 사례 |
|---|---|
| 고친 코드가 다른 곳을 망가뜨렸는지 바로 안다 | 인용 검사의 공백 정리 규칙을 바꿨더니, 줄바꿈이 섞인 원문을 정확히 인용한 답까지 "원문에 없는 문구"로 판정된다. 기존 테스트가 실패해서 병합 전에 알게 된다 |
| 사람이 매번 손으로 확인하지 않아도 된다 | 변경 요청마다 CI(GitHub Actions)가 테스트를 돌리고, 실패하면 병합을 막는다 |
| 코드가 지켜야 할 규칙이 문서로 남는다 | 새로 온 팀원이 테스트 파일만 읽어도 "없는 근거 ID면 이렇게 판정한다" 같은 규칙을 안다 |

**학습 목표**

- 이름 규칙에 맞춰 `assert` 테스트를 쓰고 pytest로 실행합니다.
- 통과·실패 출력과 실패 메시지를 읽고 어디가 틀렸는지 찾습니다.
- `parametrize`로 여러 입력을 한 테스트로 확인하고, fixture로 준비물(데이터, DB 연결)을 넘기고 정리합니다.
- 종료 코드로 결과를 다른 프로그램(CI)에 넘기는 원리를 확인합니다.
- 챗봇 코드 가운데 LLM 호출 앞뒤의 코드(문맥 만들기, 인용 확인)를 가짜 모델로 테스트합니다.

<img src="../images/ap_00_pytest_flow.png" width="1050" alt="pytest가 test_로 시작하는 파일과 함수를 찾아 하나씩 실행하고, assert 조건이 참이면 PASSED, 거짓이면 FAILED로 판정한 뒤 모두 통과하면 종료 코드 0, 하나라도 실패하면 1을 돌려주는 순서입니다.">

## 1. pytest와 연습 파일

**pytest**는 파이썬에서 가장 널리 쓰는 테스트 도구입니다. 테스트는 `assert 조건` 한 줄로 쓰고, 조건이 거짓이면 pytest가 실제 값과 기대값이 어떻게 다른지 보여 줍니다.

**무엇을 단위 테스트로 만들까**

단위 테스트(unit test)는 함수 하나에 정한 입력을 넣고 결과가 맞는지 확인하는 테스트입니다. 다음 기준에 맞는 코드부터 만듭니다.

- **같은 입력에 늘 같은 결과를 내는 코드:** 맞는 결과를 미리 적어 두고 `assert`로 비교할 수 있습니다.
- **틀려도 오류 없이 값만 틀리는 코드:** 실행은 되는데 결과가 틀리면 사람이 알아채기 어렵습니다.
- **자주 고치는 코드와 버그가 났던 코드:** 고친 뒤 같은 버그가 다시 생기지 않게 막습니다.
- **LLM이 만드는 답은 단위 테스트로 판정하지 않습니다:** 같은 질문에도 답이 달라지므로, 답의 품질은 교안 01·02처럼 골든셋 평가와 회귀 게이트로 봅니다. 단위 테스트는 LLM을 부르기 전후의 코드에 씁니다.

예를 들면 다음과 같습니다.

- **일반 코드:** 날짜·금액 계산, 문자열 정규화, 입력값 검사, 설정 파일 읽기
- **에이전트·RAG 코드**
    - 프롬프트 문맥 만들기: 검색 문서를 근거 ID와 함께 이어 붙이는 함수
    - 답변 검사: 답변에서 근거 ID 뽑기, 인용이 원문과 맞는지 대조, 구조화 출력(Pydantic 모델)의 필수 값 확인
    - 그래프 분기: LangGraph 조건부 엣지 함수(State를 받아 다음 노드 이름을 돌려줌), 반복 횟수 제한
    - 도구 본문: `@tool` 함수가 하는 DB 조회와 계산(모델 없이 함수를 직접 불러 확인)

이 부록의 연습 파일도 이 기준을 따릅니다. 인용 대조(`citation_check.py`)와 LangChain 답변 코드에서 LLM이 아닌 부분(`rag_answer.py`)을 테스트합니다.

이 노트북과 같은 폴더의 연습 파일입니다. 관례대로 테스트할 코드는 폴더 맨 위에, 테스트 파일은 `tests/` 폴더에 둡니다.

```text
부록_01_pytest_기초/
├── citation_check.py          테스트할 함수(교안 01의 normalize, check_citation)
├── rag_answer.py              함께 따라하기에서 테스트할 LangChain 답변 코드
├── library.db                 대출 규칙 표가 든 SQLite DB
├── pytest.ini                 pytest 설정(테스트 폴더, import 경로, DeepEval 플러그인 끄기)
└── tests/
    ├── test_citation_check.py citation_check.py 의 테스트
    ├── test_failure_demo.py   실패 출력을 읽어 보려고 일부러 틀리게 쓴 테스트
    ├── test_fixture_db.py     fixture로 library.db 에 연결하고 닫는 테스트
    └── test_followalong.py    함께 따라하기에서 작성하는 테스트
```

`.py` 파일마다 맨 위 설명에 무엇을 하는 파일인지와 터미널 실행 명령을 적어 두었습니다.

아래 셀은 노트북에서 pytest를 실행하는 도구 `run_pytest`를 만듭니다. 터미널 명령 `pytest 옵션 파일`을 대신 실행할 뿐이니 안의 코드는 자세히 읽지 않아도 됩니다.

In [ ]:
import subprocess
import sys


def run_pytest(*args):
    """이 노트북 폴더에서 pytest 를 실행하고 종료 코드와 출력을 보여 줍니다."""
    # 노트북과 같은 가상환경의 pytest 를 실행합니다.
    # --color=no: 출력에 색 코드가 섞이지 않게 합니다.
    command = [sys.executable, "-m", "pytest", "--color=no", *args]
    completed = subprocess.run(command, capture_output=True, text=True)
    print("종료 코드:", completed.returncode)
    print(completed.stdout)

## 2. 첫 테스트 쓰고 실행하기

pytest는 정해진 이름의 파일과 함수만 테스트로 찾습니다.

- **파일**: `test_`로 시작하는 파일
- **함수**: 그 파일 안에서 `test_`로 시작하는 함수
- **판정**: 함수 안의 `assert 조건`이 거짓이면 실패, 끝까지 오류 없이 실행되면 통과

`test_citation_check.py`의 앞부분입니다. `normalize`를 세 가지 입력으로 확인합니다.

```python
"""citation_check.py 의 테스트입니다. test_ 로 시작하는 함수 하나가 테스트 하나입니다.

실행 방법(부록_01_pytest_기초 폴더에서): uv run pytest -v tests/test_citation_check.py
"""
import pytest

from citation_check import check_citation, normalize


# 1) assert 로 쓴 기본 테스트: normalize 가 뜻과 상관없는 표기 차이를 하나로 맞추는지 확인합니다.
#    assert 실행 결과 == 기대값: 왼쪽에 함수가 돌려준 값, 오른쪽에 정답으로 정한 값을 쓰는 것이 관례입니다.
#    실패하면 pytest 가 이 순서로 차이를 보여 줍니다(- 는 기대값, + 는 실행 결과).
def test_line_break_becomes_space():
    # 줄바꿈은 공백 하나가 됩니다.
    assert normalize("일반 도서를\n14일 동안") == "일반 도서를 14일 동안"


def test_many_spaces_become_one():
    assert normalize("1인   7권까지") == "1인 7권까지"


def test_middle_dot_shapes_are_unified():
    # 모양이 다른 가운뎃점(･, ∙)이 모두 "·" 하나로 바뀌어야 합니다.
    assert normalize("대출･반납∙연장") == "대출·반납·연장"
```

In [ ]:
run_pytest("tests/test_citation_check.py")

기본 출력은 테스트마다 글자 하나를 찍습니다. `.`은 통과, `F`는 실패입니다.

`-v`(verbose)를 붙이면 테스트마다 이름과 `PASSED`·`FAILED`가 한 줄씩 나옵니다.

In [ ]:
run_pytest("-v", "tests/test_citation_check.py")

터미널에서는 저장소 최상위 폴더에서 이 폴더로 이동해 실행합니다.

```bash
# 저장소 최상위 폴더에서 이 부록 폴더로 이동합니다.
cd day61_평가자동화_MLflow/부록_01_pytest_기초

# uv run  : 실습 환경에 설치된 pytest 로 실행합니다.
# pytest  : 테스트를 찾아 실행하는 프로그램입니다.
# -v      : 테스트마다 이름과 결과(PASSED, FAILED)를 한 줄씩 출력합니다.
# tests/test_citation_check.py : 실행할 테스트 파일입니다. 경로를 빼면 pytest.ini 의 testpaths(tests 폴더)를 모두 실행합니다.
uv run pytest -v tests/test_citation_check.py
```

실행할 범위는 경로로 정합니다.

| 명령 | 실행하는 테스트 |
|---|---|
| `uv run pytest tests` | `tests` 폴더의 모든 테스트 파일 |
| `uv run pytest` | 위와 같음(`pytest.ini`의 `testpaths = tests`) |
| `uv run pytest tests/test_citation_check.py` | 파일 하나 |
| `uv run pytest tests/test_citation_check.py::test_many_spaces_become_one` | 테스트 함수 하나 |

<img src="../images/ap_10_pytest_pass.png" width="1050" alt="터미널에서 uv run pytest -v tests/test_citation_check.py 를 실행해 테스트 7개가 모두 PASSED로 나오고 마지막 줄에 7 passed가 찍힌 화면입니다.">

## 3. 실패한 테스트 읽기

`test_failure_demo.py`는 일부러 틀린 기대값을 적은 테스트입니다. `normalize`는 줄바꿈을 공백으로 바꾸는데, 이 테스트는 줄바꿈이 그대로 남기를 기대합니다.

```python
"""실패한 테스트의 출력을 읽어 보려고 일부러 틀린 기대값을 적은 파일입니다.

실행 방법(부록_01_pytest_기초 폴더에서): uv run pytest tests/test_failure_demo.py
"""
from citation_check import normalize


def test_keeps_line_break():
    # 일부러 틀린 기대값: normalize 는 줄바꿈을 공백으로 바꾸므로 이 테스트는 실패합니다.
    result = normalize("일반 도서를\n14일 동안")
    assert result == "일반 도서를\n14일 동안", "normalize 결과가 원래 문장과 다릅니다"
```

In [ ]:
run_pytest("tests/test_failure_demo.py")

실패 출력은 세 부분으로 읽습니다.

- **`FAILURES` 구역**: `>`가 실패한 줄을 가리키고, `E`로 시작하는 줄에 메시지와 두 값의 차이가 나옵니다(실제 값을 `==` 왼쪽에 쓰면 `-`는 기대값, `+`는 실제 값).
- **`short test summary info`**: 실패한 테스트 이름과 메시지를 한 줄씩 모아 보여 줍니다. 실패가 많을 때는 여기부터 읽습니다.
- **마지막 줄**: `1 failed`처럼 개수를 요약합니다.

<img src="../images/ap_11_pytest_fail.png" width="1050" alt="터미널에서 uv run pytest tests/test_failure_demo.py 를 실행해 실패한 줄을 가리키는 > 표시, 두 문자열의 차이를 보여 주는 E 줄, short test summary info, 1 failed 가 보이는 화면입니다.">

자주 쓰는 옵션입니다.

| 옵션 | 원래 이름 | 하는 일 |
|---|---|---|
| `-v` | `--verbose`(자세히) | 테스트마다 이름과 결과를 한 줄씩 |
| `-q` | `--quiet`(조용히) | 테스트마다 한 글자(`.`, `F`)와 마지막 개수 요약만 |
| `--tb=short` | traceback(실패 위치 추적) | 실패한 줄과 `E` 줄만 짧게(`--tb=no`는 생략) |
| `-k 이름일부` | keyword(키워드) | 이름에 그 글자가 든 테스트만 |

이름에 `middle_dot`이 든 테스트만 골라 실행합니다.

In [ ]:
run_pytest("-v", "-k", "middle_dot", "tests/test_citation_check.py")

## 4. parametrize와 fixture

테스트를 여러 개 쓰다 보면 두 가지가 반복됩니다. 같은 확인을 입력만 바꿔 여러 번 하는 일과, 테스트마다 같은 준비(데이터, 파일, DB 연결)를 하는 일입니다. pytest는 이 둘을 parametrize와 fixture로 줄여 줍니다.

| | parametrize | fixture |
|---|---|---|
| 무엇인가 | 테스트 함수에 붙이는 입력·기대값 목록 | 테스트에 필요한 준비물을 만드는 함수 |
| pytest가 하는 일 | 목록의 줄마다 테스트를 따로 실행하고 따로 판정 | 테스트 전에 불러, 돌려준 값을 테스트 인자로 넘김 |
| 쓰는 법 | `@pytest.mark.parametrize("인자 이름", [값 목록])` | `@pytest.fixture`를 붙인 함수의 이름을 테스트 인자에 적음 |
| 테스트 수 | 목록 줄 수만큼 늘어남 | 그대로 |

### 4-1. 테스트 파일에서 보기

`test_citation_check.py`의 뒷부분입니다. 인용 검사 네 경우는 parametrize로, 함께 쓰는 원문 사전은 fixture로 넘깁니다.

```python
# 2) fixture: 인용 검사 테스트들이 함께 쓰는 원문 사전을 테스트마다 새로 만들어 넘깁니다.
@pytest.fixture
def source_texts():
    """근거 ID -> 원문 사전입니다. 줄바꿈이 섞인 원문으로 정규화까지 확인합니다."""
    return {
        "loan-basic": "회원은 일반 도서를 1인 7권까지\n14일 동안 빌릴 수 있습니다.",
        "reading-room": "열람실 좌석은 4시간 단위로 예약합니다.",
    }


# 3) parametrize: 입력만 바꿔 같은 확인을 네 번 합니다. 줄마다 따로 판정합니다.
@pytest.mark.parametrize(
    "evidence_id, quote, expected",
    [
        ("loan-basic", "1인 7권까지 14일 동안", "확인"),
        ("loan-weekend", "주말에는 3일 더", "없는 근거 ID"),
        ("reading-room", "4시간 단위", "검색하지 않은 문서"),
        ("loan-basic", "1인 7권까지 30일 동안", "원문에 없는 문구"),
    ],
    ids=["ok", "unknown_id", "not_searched", "changed_number"],
)
def test_check_citation(source_texts, evidence_id, quote, expected):
    citation = {"evidence_id": evidence_id, "quote": quote}
    # 이번에 검색한 문서는 loan-basic 하나입니다.
    assert check_citation(citation, source_texts, ["loan-basic"]) == expected
```

<img src="../images/ap_00_parametrize_fixture.png" width="1050" alt="pytest는 fixture source_texts()가 실행마다 새로 만든 원문 사전과 parametrize 목록의 한 줄을 인자로 넣어 test_check_citation을 ok, unknown_id, not_searched, changed_number 네 번 실행합니다. 줄마다 기대값이 다르고 네 번 모두 통과합니다.">

pytest는 테스트 함수의 인자를 **이름으로** 찾아 채웁니다.

| 인자 | 채우는 것 | 값 |
|---|---|---|
| `evidence_id`, `quote`, `expected` | parametrize 목록의 한 줄 | 줄마다 다른 입력과 기대값 |
| `source_texts` | 같은 이름의 fixture `source_texts()` | 근거 ID -> 원문 사전 |

그래서 `test_check_citation`은 목록 줄 수만큼 네 번 실행되고, 첫 번째(`[ok]`) 실행은 아래처럼 부른 것과 같습니다.

```python
test_check_citation(source_texts={"loan-basic": "...", "reading-room": "..."},
                    evidence_id="loan-basic", quote="1인 7권까지 14일 동안", expected="확인")
```

**`ids`**: 목록의 줄마다 붙이는 이름입니다. 출력에 `test_check_citation[changed_number]`처럼 나오고, `-k changed_number`로 그 줄만 다시 실행할 수 있습니다. 빼면 pytest가 값을 이어 이름을 만드는데, 한글은 `[loan-basic-1\uc778 7\uad8c...]`처럼 글자 코드로 바뀌어 읽기 어렵습니다.

In [ ]:
run_pytest("-v", "-k", "check_citation", "tests/test_citation_check.py")

### 4-2. fixture는 테스트마다 새로 만든다

pytest는 한 파일의 테스트를 **적힌 순서대로** 실행합니다. 그래서 여러 테스트가 객체 하나를 나눠 쓰면, 앞 테스트가 고친 내용이 뒤 테스트로 넘어갑니다.

**fixture 없이**: 파일 맨 위에 사전 하나를 두고 두 테스트가 나눠 씁니다.

```python
SOURCE_TEXTS = {"loan-basic": "회원은 일반 도서를 1인 7권까지 14일 동안 빌릴 수 있습니다."}  # 모든 테스트가 나눠 쓰는 사전 하나


def test_changed_rule():
    # 규칙이 바뀐 경우를 시험하려고 원문을 고칩니다.
    SOURCE_TEXTS["loan-basic"] = "회원은 일반 도서를 1인 7권까지 21일 동안 빌릴 수 있습니다."
    assert check_citation({"evidence_id": "loan-basic", "quote": "21일 동안"}, SOURCE_TEXTS, ["loan-basic"]) == "확인"


def test_ok_citation():
    # 앞 테스트가 고친 사전을 그대로 받습니다.
    assert check_citation({"evidence_id": "loan-basic", "quote": "14일 동안"}, SOURCE_TEXTS, ["loan-basic"]) == "확인"
```

두 테스트를 함께 돌리면 `test_ok_citation`이 "원문에 없는 문구"로 실패합니다(1 failed, 1 passed). 이 테스트만 따로 돌리면 통과하므로, `check_citation`에는 문제가 없는데 앞 테스트 때문에 실패한 것입니다.

**fixture로**: 같은 두 테스트가 사전을 fixture에서 받게 바꿉니다.

```python
@pytest.fixture
def source_texts():
    return {"loan-basic": "회원은 일반 도서를 1인 7권까지 14일 동안 빌릴 수 있습니다."}


def test_changed_rule(source_texts):
    # 이 테스트가 받은 사전만 바뀝니다.
    source_texts["loan-basic"] = "회원은 일반 도서를 1인 7권까지 21일 동안 빌릴 수 있습니다."
    assert check_citation({"evidence_id": "loan-basic", "quote": "21일 동안"}, source_texts, ["loan-basic"]) == "확인"


def test_ok_citation(source_texts):
    # pytest 가 source_texts() 를 다시 불러 만든 새 사전을 받으므로 원문은 처음 그대로입니다.
    assert check_citation({"evidence_id": "loan-basic", "quote": "14일 동안"}, source_texts, ["loan-basic"]) == "확인"
```

이번에는 같은 순서로 돌려도 두 테스트가 모두 통과합니다(2 passed).

테스트는 이렇게 서로 **독립**이어야 합니다. 하나만 골라 돌리든(`-k`) 순서가 바뀌든 결과가 같아야, 실패했을 때 그 테스트가 확인한 코드만 보면 됩니다. 앞 테스트의 결과를 다음 테스트로 넘기는 "이어지는 테스트"는 이 원칙을 깨므로, 단계가 이어지는 흐름은 한 테스트 함수 안에서 차례로 확인합니다.

그렇다고 모든 준비물을 fixture로 만들 필요는 없습니다. 기준은 **테스트가 그 값을 고칠 수 있는가**입니다.

| 준비물 | 두는 곳 | 예 |
|---|---|---|
| 고칠 수 없는 값 | 파일 맨 위 상수 | 질문 문자열, 기대값 문자열, 숫자, 튜플 |
| 고칠 수 있는 값 | fixture(테스트마다 새로 만듦) | 사전, 리스트, `Document` 같은 객체 |

이름을 대문자로 쓰는 것은 관례일 뿐이라, 위 `SOURCE_TEXTS`처럼 값이 사전이면 테스트나 테스트하는 함수가 얼마든지 고칠 수 있습니다.

### 4-3. yield로 준비하고 정리하기: DB 연결

DB 연결처럼 쓰고 나면 닫아야 하는 준비물은 `return` 대신 `yield`로 넘깁니다. `yield` 앞은 테스트 전에 하는 준비, 뒤는 테스트가 끝난 뒤 하는 정리입니다. 정리 코드는 테스트가 실패해도 실행되므로, 테스트 안에서 직접 닫다가 `assert`에서 멈춰 연결이 남는 일이 없습니다.

이 폴더의 `library.db`에는 대출 규칙 표 `loan_rules`(자료 종류, 1인 최대 권수, 대출 일수)가 세 줄 들어 있습니다. `test_fixture_db.py`의 앞부분은 이 DB에 연결하는 fixture와, 그 연결로 규칙을 읽는 테스트 두 개입니다.

```python
"""fixture 로 도서관 DB(library.db)에 연결해 테스트에 넘기고, 테스트가 끝나면 연결을 닫습니다.

실행 방법(부록_01_pytest_기초 폴더에서): uv run pytest -v -s tests/test_fixture_db.py
-s 는 fixture 안의 print 를 화면에 보여 주는 옵션입니다.
"""
import sqlite3

import pytest


# 1) 기본 범위(function): 테스트마다 연결을 새로 열어 넘기고, 테스트가 끝나면 닫습니다.
@pytest.fixture
def library_db():
    # 준비: 이 폴더의 library.db 에 연결합니다.
    conn = sqlite3.connect("library.db")
    print("\n[library_db] 연결 열기")

    # yield: 연결을 테스트에 넘기고, 테스트가 끝날 때까지 여기서 기다립니다.
    yield conn

    # 정리: 테스트가 통과하든 실패하든 실행됩니다.
    conn.close()
    print("\n[library_db] 연결 닫기")


def test_library_general_loan_days(library_db):
    row = library_db.execute("SELECT loan_days FROM loan_rules WHERE book_type = '일반 도서'").fetchone()
    assert row[0] == 14


def test_library_rule_count(library_db):
    assert library_db.execute("SELECT COUNT(*) FROM loan_rules").fetchone()[0] == 3
```

`-s`는 fixture 안의 `print`를 화면에 보여 주는 옵션입니다. 이름에 `library`가 든 두 테스트만 실행합니다.

In [ ]:
run_pytest("-v", "-s", "-k", "library", "tests/test_fixture_db.py")

테스트마다 `연결 열기 -> PASSED -> 연결 닫기`가 한 번씩 나옵니다. 기본 범위라서 pytest가 테스트마다 fixture를 다시 불러 연결을 새로 열고, 테스트가 끝나면 `yield` 뒤의 정리 코드로 닫습니다.

### 4-4. scope: 준비물을 얼마나 자주 새로 만들까

fixture는 기본으로 테스트마다 새로 만듭니다. 벡터 저장소나 임베딩 모델처럼 불러오는 데 오래 걸리는 준비물은 `@pytest.fixture(scope="module")`처럼 범위를 넓혀 한 번만 만들고 나눠 씁니다.

| scope | 새로 만드는 단위 | 알맞은 준비물 |
|---|---|---|
| `function`(기본) | 테스트 함수 하나마다 | 테스트가 고치는 것: 고칠 사전·목록 |
| `class` | 테스트 클래스 하나마다 | 클래스로 묶은 테스트가 함께 읽는 것 |
| `module` | 테스트 파일 하나마다 | 파일 안에서 읽기만 하는 것: 벡터 저장소, 읽기 전용 DB 연결 |
| `package` | 테스트 폴더(패키지) 하나마다 | 폴더 전체가 읽기만 하는 것 |
| `session` | `pytest` 실행 한 번마다 | 전체가 함께 쓰는 것: 테스트용 서버 연결 |

`test_fixture_db.py`의 뒷부분입니다. 같은 `library.db`를 이번에는 파일당 한 번만 열어 두 테스트가 나눠 씁니다.

```python
# 2) module 범위: 이 파일에서 연결을 한 번만 열고, 파일 안의 테스트가 그 연결을 나눠 씁니다.
#    한 테스트가 DB 를 고치면 다음 테스트도 고친 DB 를 보게 되므로, 읽기만 하는 테스트에 씁니다.
@pytest.fixture(scope="module")
def shared_db():
    conn = sqlite3.connect("library.db")
    print("\n[shared_db] 연결 열기")

    yield conn

    conn.close()
    print("\n[shared_db] 연결 닫기")


def test_shared_new_book_days(shared_db):
    row = shared_db.execute("SELECT loan_days FROM loan_rules WHERE book_type = '신간 코너 도서'").fetchone()
    assert row[0] == 7


def test_shared_new_book_max(shared_db):
    row = shared_db.execute("SELECT max_books FROM loan_rules WHERE book_type = '신간 코너 도서'").fetchone()
    assert row[0] == 3
```

In [ ]:
run_pytest("-v", "-s", "-k", "shared", "tests/test_fixture_db.py")

`연결 열기`와 `연결 닫기`가 테스트 두 개에 걸쳐 한 번씩만 나옵니다. 두 테스트가 같은 연결을 썼다는 뜻입니다.

범위를 넓힌 준비물은 여러 테스트가 나눠 쓰므로 테스트에서 고치지 않습니다. 한 테스트가 DB에 행을 넣거나 지우면, 뒤 테스트가 고친 DB를 보게 되어 fixture 없이 사전을 나눠 쓸 때와 같은 문제가 다시 생깁니다.

## 5. 종료 코드로 결과 넘기기

pytest는 끝날 때 결과를 숫자(종료 코드)로 돌려줍니다. CI 같은 다른 프로그램은 출력을 읽지 않고 이 숫자로 성공과 실패를 가립니다(부록 02).

| 종료 코드 | 뜻 |
|---|---|
| 0 | 모든 테스트 통과 |
| 1 | 하나 이상 실패 |
| 5 | 실행할 테스트가 하나도 없음 |

In [ ]:
# -q(--quiet): 테스트마다 줄을 찍지 않고 결과 요약만 짧게 보여 줍니다. 세 경우 모두 종료 코드만 보면 되므로 씁니다.

# 1) 통과: 테스트 7개가 모두 통과해 종료 코드 0
run_pytest("-q", "tests/test_citation_check.py")

# 2) 실패: 일부러 틀린 테스트라 종료 코드 1
#    --tb=no: 실패한 코드 위치와 E 줄(traceback)을 생략하고 실패한 테스트 이름만 남깁니다.
run_pytest("-q", "--tb=no", "tests/test_failure_demo.py")

# 3) 테스트 없음: 종료 코드 5
#    -k no_such_test: 이름에 "no_such_test"가 든 테스트만 고르는데 그런 테스트가 없어 하나도 실행되지 않습니다.
run_pytest("-q", "-k", "no_such_test", "tests/test_citation_check.py")

## 6. 🖐️ 함께 따라하기: LangChain 답변 코드 테스트하기

챗봇 코드 가운데 LLM을 부르기 전후의 코드(문맥 만들기, 분기, 인용 확인)를 단위 테스트로 확인합니다. 이 코드들은 입력이 같으면 결과가 늘 같아서 `assert`로 판정할 수 있습니다.

`rag_answer.py`는 도서관 안내 챗봇에서 그 부분입니다.

- `format_documents(docs)`: 검색한 `Document`들을 `[근거 ID] 본문` 줄로 이어 프롬프트 문맥을 만듭니다.
- `answer_question(question, docs, llm)`: 검색한 문서가 없으면 모델을 부르지 않고 안내 문장(`ABSTAIN`)을 돌려줍니다. 있으면 `answer_prompt | llm | StrOutputParser()` 체인으로 답한 뒤, 답이 검색하지 않은 문서를 인용했으면(모델이 지어낸 근거 ID) `ABSTAIN`으로 바꿉니다. 인용이 없는 답은 그대로 돌려줍니다.

단위 테스트에서는 진짜 모델 대신 LangChain이 테스트용으로 제공하는 가짜 모델 `FakeListChatModel`을 넣습니다. 가짜 모델은 "모델이 이렇게 답했다면"을 정해 두는 도구라, 테스트하는 대상은 모델이 아니라 **모델의 답을 받아 처리하는 우리 코드**입니다. 진짜 모델은 지어낸 인용을 언제 낼지 모르지만, 가짜 모델로는 그런 답을 일부러 넣어 인용 확인 코드가 막는지 매번 같은 결과로 확인할 수 있습니다.

`tests/test_followalong.py`를 편집기(VS Code 등)로 열어 테스트를 씁니다. 같은 확인을 fixture·parametrize **없이** 한 번, **써서** 한 번 작성해 두 방식을 비교합니다. 테스트마다 위에 단 주석에 무엇을 만들고 어떻게 쓰는지 적어 두었으니, `pytest.fail("이 테스트를 작성하세요")` 줄을 지우고 그 자리에 코드를 씁니다.

작성할 파일의 처음 모습입니다.

```python
"""함께 따라하기: 도서관 안내 챗봇 코드(rag_answer.py)의 테스트를 씁니다.

같은 확인을 fixture·parametrize 없이 한 번, 쓰고 한 번 작성해 두 방식을 비교합니다.
LLM 은 부르지 않습니다. 모델 자리에는 LangChain 이 테스트용으로 제공하는 가짜 모델 FakeListChatModel 을 넣어,
"모델이 이렇게 답했다면" 우리 코드가 어떻게 처리하는지 확인합니다.
실행 방법(부록_01_pytest_기초 폴더에서): uv run pytest -v tests/test_followalong.py
"""
import pytest
from langchain_core.documents import Document
from langchain_core.language_models.fake_chat_models import FakeListChatModel

from rag_answer import ABSTAIN, answer_question, format_documents

QUESTION = "일반 도서는 몇 권까지 빌릴 수 있나요?"
# format_documents 가 문서 두 개로 만들어야 할 문맥입니다.
EXPECTED_CONTEXT = ("[loan-basic] 회원은 일반 도서를 1인 7권까지 14일 동안 빌릴 수 있습니다.\n\n"
                    "[loan-extension] 빌린 도서는 1회에 한해 7일 연장할 수 있습니다.")

# 가짜 모델이 돌려줄 답입니다. 검색한 문서는 loan-basic, loan-extension 두 개이고, loan-weekend 는 검색하지 않은 문서입니다.
# 앞의 둘은 검색한 문서만 인용했고, 뒤의 둘은 검색하지 않은 문서를 인용했습니다.
CITED = "일반 도서는 1인 7권까지 14일 동안 빌릴 수 있습니다. [loan-basic]"
TWO_CITED = "일반 도서는 14일 동안 빌리고 1회 7일 연장할 수 있습니다. [loan-basic] [loan-extension]"
UNKNOWN_ID = "주말에는 3일 더 빌릴 수 있습니다. [loan-weekend]"
MIXED = "일반 도서는 14일 동안 빌리고 주말에는 3일 더 빌릴 수 있습니다. [loan-basic] [loan-weekend]"


# 1-1. [fixture 없이] format_documents 가 문서 두 개로 EXPECTED_CONTEXT 를 만드는지 확인하는 테스트
#      - 테스트 안에서 Document 두 개를 직접 만들어 docs 목록에 담습니다.
#        Document(page_content="회원은 일반 도서를 1인 7권까지 14일 동안 빌릴 수 있습니다.", metadata={"source_id": "loan-basic"})
#        Document(page_content="빌린 도서는 1회에 한해 7일 연장할 수 있습니다.", metadata={"source_id": "loan-extension"})
#      - format_documents(docs) 가 EXPECTED_CONTEXT 와 같은지 assert 합니다.
def test_format_documents_plain():
    pytest.fail("이 테스트를 작성하세요")


# 1-2. [fixture 만들기] 1-1의 문서 두 개를 돌려주는 fixture
#      - 지금은 빈 목록을 돌려줘서 docs 를 쓰는 테스트가 실패합니다.
#      - 1-1에서 만든 Document 두 개를 담은 목록을 돌려주도록 고칩니다.
@pytest.fixture
def docs():
    return []


# 1-3. [fixture 쓰기] 1-1과 같은 확인을, 문서를 인자 docs 로 받아서 하는 테스트
#      - 문서를 만드는 코드 없이 assert 한 줄만 씁니다.
def test_format_documents_with_fixture(docs):
    pytest.fail("이 테스트를 작성하세요")


# 2-1. [parametrize 없이] answer_question 의 인용 확인이 근거 있는 답은 통과시키고, 지어낸 인용은 막는지 확인하는 테스트
#      - 가짜 모델 FakeListChatModel(responses=[답]) 은 불리면 그 답을 돌려줍니다. 모델이 그렇게 답한 상황을 만드는 것입니다.
#      - 두 경우를 한 함수 안에서 차례로 확인합니다.
#        모델 답이 CITED 이면 answer_question(QUESTION, docs, 가짜 모델) 결과가 CITED 그대로인지
#        모델 답이 UNKNOWN_ID(검색하지 않은 loan-weekend 인용)이면 결과가 ABSTAIN 인지
def test_citation_guard_plain(docs):
    pytest.fail("이 테스트를 작성하세요")


# 2-2. [parametrize 쓰기] 같은 확인을 모델 답 네 가지로 하나씩 따로 판정하는 테스트
#      - 함수 위에 아래 parametrize 를 붙이고, 인자를 (docs, reply, expected) 로 고칩니다.
#        @pytest.mark.parametrize(
#            "reply, expected",
#            [(CITED, CITED), (TWO_CITED, TWO_CITED), (UNKNOWN_ID, ABSTAIN), (MIXED, ABSTAIN)],
#            ids=["cited", "two_cited", "unknown_id", "mixed"],
#        )
#      - 모델 답이 reply 일 때 answer_question 결과가 expected 인지 assert 한 줄로 확인합니다.
def test_citation_guard(docs):
    pytest.fail("이 테스트를 작성하세요")
```

In [ ]:
run_pytest("-v", "tests/test_followalong.py")

테스트 일곱 개(parametrize 네 줄 포함)가 모두 `PASSED`이면 완성입니다. 출력에서 짝지은 두 버전을 비교합니다.

- **fixture 없이(1-1)와 쓴 것(1-3)**: 결과는 같지만 1-1은 문서 목록을 테스트 안에 직접 써서, 문서가 필요한 테스트가 늘 때마다 같은 목록을 다시 써야 합니다. fixture는 한 곳에서 만들고 테스트마다 새로 넘깁니다.
- **parametrize 없이(2-1)와 쓴 것(2-2)**: 2-1은 출력에 한 줄로 나오고 첫 `assert`가 실패하면 둘째 경우는 확인되지 않습니다. 2-2는 `[cited]`, `[unknown_id]`처럼 네 줄로 나와 경우마다 따로 판정됩니다.

테스트가 정말 우리 코드를 검사하는지는 코드를 망가뜨려 보면 압니다. `answer_question`에서 인용 확인 부분을 지우면 지어낸 인용을 확인하는 `test_citation_guard_plain`, `[unknown_id]`, `[mixed]`가 실패합니다. `[mixed]`는 인용 둘 중 하나만 지어낸 경우라, 첫 인용만 확인하도록 잘못 고쳐도 이 테스트가 잡습니다.

## 7. 전체 테스트 한 번에 실행하기

지금까지는 파일을 하나씩 골라 실행했습니다. 실무에서는 코드를 고친 뒤 `tests` 폴더 전체를 한 번에 실행하고, CI(부록 02)도 변경 요청마다 이렇게 폴더 전체를 실행합니다.

터미널에서는 `uv run pytest tests`입니다. `-q`로 짧게, `--tb=no`로 실패 위치 출력 없이 결과만 봅니다.

In [ ]:
run_pytest("-q", "--tb=no", "tests")

따라하기까지 완성했다면 일부러 틀리게 쓴 `test_failure_demo.py` 하나만 실패해 `1 failed, 18 passed`, 종료 코드 1이 나옵니다. 실무 저장소에는 이런 일부러 틀린 테스트를 두지 않으므로, 전체 실행이 모두 통과(종료 코드 0)해야 변경을 병합합니다.

**`pytest.ini`의 역할**: 폴더 맨 위에 두는 pytest 설정 파일입니다. pytest는 실행할 때마다 이 파일을 먼저 읽어 아래 설정을 적용하고, 출력 윗부분의 `configfile: pytest.ini` 줄이 읽었다는 표시입니다.

| 설정 | 하는 일 |
|---|---|
| `testpaths = tests` | 경로 없이 `pytest`만 실행하면 `tests` 폴더에서 테스트를 찾습니다 |
| `pythonpath = .` | `tests` 안의 테스트가 폴더 맨 위의 `citation_check.py`, `rag_answer.py`를 `import`할 수 있게 합니다 |
| `addopts = -p no:deepeval` | 모든 실행에 이 옵션을 붙입니다. 설치된 DeepEval 플러그인을 꺼 출력에 플러그인 메시지가 섞이지 않게 합니다 |